# Лабораторна робота №2
## ШІ в аналізі даних для енергетики

**Персональний варіант:** енергетика / програмне забезпечення / візуалізація.

Модель машинного навчання класифікує енергоспоживання наступної години як звичайне або високе. Використано реальні дані UCI з першої лабораторної. Відгуки й CountVectorizer з типової інструкції замінено задачею персонального варіанта. Ключове обмеження: набір описує одне домогосподарство у Франції в 2006–2010 роках.

### 1. Підготовка середовища й даних

Покладіть hourly_clean.csv у data/ поряд із цим ноутбуком. Таблиця містить лише повні години; asfreq повертає часові пропуски в сітку, щоб лаги не перескакували через розриви. Мітка «високе» визначається порогом 75-го процентиля енергії наступної години, обчисленим лише на тренувальній частині.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "main.py").exists():
    ROOT = Path(".").resolve()
    assert (ROOT / "main.py").exists(), "Відкрийте ноутбук із папки лабораторної 2"
sys.path.insert(0, str(ROOT))
from main import run_lab
metrics = run_lab()
display(pd.DataFrame([metrics["classification"]]).drop(columns=["report", "confusion_matrix", "confusion_matrix_labels"]))

### 2. Оцінювання моделі

Логістична регресія навчається на часових ознаках: вимірюваннях поточної години, лагах та календарних ознаках. Дані поділено в хронологічному порядку 70/30. Для порівняння наведено базовий прогноз найчастішого класу. Accuracy не описує пропуски високих навантажень, тому також аналізуються precision, recall, F1 і матриця помилок.

In [ ]:
print("Навчальні / тестові рядки:", metrics["split"]["train_rows"], "/", metrics["split"]["test_rows"])
print("Матриця помилок (рядки — факт, стовпці — прогноз):")
display(pd.DataFrame(metrics["classification"]["confusion_matrix"], index=["Факт: звичайне", "Факт: високе"], columns=["Прогноз: звичайне", "Прогноз: високе"]))
for filename in ["accuracy.png", "class_distribution.png", "confusion_matrix.png"]:
    display(Image(filename=str(ROOT / "results" / "plots" / filename)))

### 3. Упередженість та етичні обмеження

Дані охоплюють лише одне домогосподарство, одну країну та історичний період. Вони не представляють інші домогосподарства, соціальні групи чи сучасні умови. Пропуски вимірювань можуть бути нерівномірними в часі. Демографічних ознак у наборі немає, тому справедливість моделі між демографічними групами оцінити неможливо. Порогова мітка — статистичне визначення високого навантаження, а не аварія чи нормативне порушення. Не слід використовувати результат як автоматичне попередження без зовнішньої перевірки.

### 4. Висновок

Модель досягає вищої точності, ніж базовий прогноз, але виявляє менше половини годин високого навантаження. Це демонструє, що для енергетичного моніторингу потрібно дивитися на метрики конкретного класу, а не лише на загальну точність. Повний звіт і команди запуску наведено в report.md та README.md.